# 06 — Predictive Analytics

**Goal:** Predict whether a travel permit case will have a long duration (top tertile, ≥ 98 days).

**Approach:**
1. Engineer a flat case-level feature table from the event log (pandas only)
2. Fit three classifiers: Logistic Regression, Random Forest, XGBoost
3. Evaluate with accuracy, ROC-AUC, and top-10 feature importances

**Important caveat:** All features are computed from the complete case history.
This is a retrospective analysis — suitable for understanding what characterises long cases,
not for real-time prediction at case start. Future work should restrict features
to those available at a chosen prefix length (e.g., first 3 events only).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.load_event_log import load_xes_log, convert_to_dataframe
from src.process_summary import get_case_durations, get_variants

DATA_PATH   = Path('../data/raw/PermitLog.xes')
FIGURES_OUT = Path('../outputs/figures')
TABLES_OUT  = Path('../outputs/tables')
FIGURES_OUT.mkdir(parents=True, exist_ok=True)
TABLES_OUT.mkdir(parents=True, exist_ok=True)

In [2]:
log = load_xes_log(DATA_PATH, legacy=True)
df  = convert_to_dataframe(log)
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True)
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)
print(f'{len(df):,} events | {df["case:concept:name"].nunique():,} cases')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7065/7065 [00:00<00:00, 8870.01it/s] 


86,581 events | 7,065 cases


## 1. Feature engineering — pandas only

All features are derived from the completed event log.
One row per case.

In [3]:
case_col = 'case:concept:name'
act_col  = 'concept:name'
time_col = 'time:timestamp'

# ── Base: case duration ────────────────────────────────────────────
dur = get_case_durations(df)[['case:concept:name','duration_days']]
feats = dur.rename(columns={'case:concept:name': 'case_id'}).copy()

# ── Target: long-duration flag (top tertile) ───────────────────────
t33 = feats['duration_days'].quantile(1/3)
t67 = feats['duration_days'].quantile(2/3)
feats['duration_bucket'] = pd.cut(
    feats['duration_days'],
    bins=[-np.inf, t33, t67, np.inf],
    labels=['short', 'medium', 'long']
)
feats['is_long'] = (feats['duration_bucket'] == 'long').astype(int)
print(f'Tertile thresholds — short: < {t33:.1f}d, long: > {t67:.1f}d')
print(feats['duration_bucket'].value_counts().sort_index())

Tertile thresholds — short: < 50.4d, long: > 101.1d
duration_bucket
short     2355
medium    2355
long      2355
Name: count, dtype: int64


In [4]:
# ── Event-count features ───────────────────────────────────────────
n_events = df.groupby(case_col).size().rename('n_events')

rejection_acts = df[act_col].str.contains('REJECTED', na=False)
n_rejections = df[rejection_acts].groupby(case_col).size().rename('n_rejections').reindex(
    df[case_col].unique(), fill_value=0)

n_reminders = df[df[act_col] == 'Send Reminder'].groupby(case_col).size().rename(
    'n_reminders').reindex(df[case_col].unique(), fill_value=0)

approval_acts = df[act_col].str.contains('APPROVED', na=False)
n_approvals = df[approval_acts].groupby(case_col).size().rename('n_approvals').reindex(
    df[case_col].unique(), fill_value=0)

has_rfp = (df[df[act_col].str.contains('Request For Payment', na=False)]
           .groupby(case_col).size() > 0).rename('has_rfp').reindex(
    df[case_col].unique(), fill_value=False)

for s in [n_events, n_rejections, n_reminders, n_approvals, has_rfp]:
    feats = feats.merge(s.reset_index().rename(columns={case_col: 'case_id'}),
                        on='case_id', how='left')
feats['has_rfp'] = feats['has_rfp'].fillna(False).astype(int)
feats[['n_rejections','n_reminders','n_approvals']] = \
    feats[['n_rejections','n_reminders','n_approvals']].fillna(0).astype(int)
print('Event-count features added.')

Event-count features added.


In [5]:
# ── Start / end activity ───────────────────────────────────────────
start_act = df.groupby(case_col)[act_col].first().rename('start_activity')
end_act   = df.groupby(case_col)[act_col].last().rename('end_activity')

feats = feats.merge(start_act.reset_index().rename(columns={case_col:'case_id'}), on='case_id', how='left')
feats = feats.merge(end_act.reset_index().rename(columns={case_col:'case_id'}),   on='case_id', how='left')

feats['starts_with_trip']   = (feats['start_activity'] == 'Start trip').astype(int)
feats['ends_with_payment']  = (feats['end_activity']   == 'Payment Handled').astype(int)
feats['ends_with_reminder'] = (feats['end_activity']   == 'Send Reminder').astype(int)
print('Start/end activity features added.')

Start/end activity features added.


In [6]:
# ── Timing sub-intervals ───────────────────────────────────────────
def first_ts(activity):
    return (df[df[act_col] == activity]
            .groupby(case_col)[time_col].min()
            .rename(activity))

ts = pd.concat([
    first_ts('Permit SUBMITTED by EMPLOYEE'),
    first_ts('Permit FINAL_APPROVED by SUPERVISOR'),
    first_ts('Permit FINAL_APPROVED by DIRECTOR'),
    first_ts('Start trip'),
    first_ts('End trip'),
    first_ts('Declaration SUBMITTED by EMPLOYEE'),
    first_ts('Payment Handled'),
], axis=1)
ts.columns = ['permit_submit','final_sup','final_dir','start_trip','end_trip','decl_submit','payment']
ts['final_approval'] = ts[['final_sup','final_dir']].min(axis=1)
ts.index.name = 'case_id'
ts = ts.reset_index()

def days_between(a, b):
    return (b - a).dt.total_seconds() / 86400

ts['permit_approval_days'] = days_between(ts['permit_submit'], ts['final_approval']).clip(lower=0)
ts['approval_to_trip_days'] = days_between(ts['final_approval'], ts['start_trip']).clip(lower=0)
ts['trip_duration_days']   = days_between(ts['start_trip'], ts['end_trip']).clip(lower=0)
ts['decl_to_payment_days'] = days_between(ts['decl_submit'], ts['payment']).clip(lower=0)

timing_cols = ['case_id','permit_approval_days','approval_to_trip_days',
               'trip_duration_days','decl_to_payment_days']
feats = feats.merge(ts[timing_cols], on='case_id', how='left')
print('Timing features added.')

Timing features added.


In [7]:
# ── Compliance group ───────────────────────────────────────────────
# Replicated from notebook 05 logic
tl = ts.set_index('case_id')
has_trip = tl['start_trip'].notna()

type_a_idx = tl[
    has_trip & (tl['permit_submit'].isna() | (tl['start_trip'] < tl['permit_submit']))
].index
type_b_idx = tl[
    has_trip & tl['permit_submit'].notna() &
    (tl['start_trip'] >= tl['permit_submit']) &
    (tl['final_approval'].isna() | (tl['start_trip'] < tl['final_approval']))
].index
compliant_idx = tl[
    has_trip & tl['final_approval'].notna() &
    (tl['start_trip'] >= tl['final_approval'])
].index

def compliance_label(case_id):
    if case_id in type_a_idx: return 'type_a'
    if case_id in type_b_idx: return 'type_b'
    if case_id in compliant_idx: return 'compliant'
    return 'no_trip'

feats['compliance'] = feats['case_id'].map(compliance_label)
print(feats['compliance'].value_counts())

compliance
compliant    5736
type_a        746
type_b        583
Name: count, dtype: int64


In [8]:
# ── Case-level attributes from XES ────────────────────────────────
case_attrs = df.drop_duplicates(case_col)[[
    case_col,
    'case:OrganizationalEntity',
    'case:RequestedBudget',
    'case:TotalDeclared',
    'case:Overspent',
]].rename(columns={case_col: 'case_id'})

feats = feats.merge(case_attrs, on='case_id', how='left')
feats['case:Overspent'] = feats['case:Overspent'].astype(int)
feats['case:RequestedBudget'] = pd.to_numeric(feats['case:RequestedBudget'], errors='coerce')
feats['case:TotalDeclared']   = pd.to_numeric(feats['case:TotalDeclared'],   errors='coerce')
feats['budget_utilisation']   = (feats['case:TotalDeclared'] / feats['case:RequestedBudget']).clip(upper=5)
print('Case attributes added.')

Case attributes added.


In [9]:
# ── Variant rank ───────────────────────────────────────────────────
variants = get_variants(df)  # sorted by frequency descending
variant_rank_map = {v: i+1 for i, v in enumerate(variants['variant'])}

case_variants = (
    df.sort_values([case_col, time_col])
    .groupby(case_col)[act_col]
    .apply(lambda x: ' -> '.join(x))
    .rename('variant')
    .reset_index()
    .rename(columns={case_col: 'case_id'})
)
case_variants['variant_rank'] = case_variants['variant'].map(variant_rank_map).fillna(9999).astype(int)
feats = feats.merge(case_variants[['case_id','variant_rank']], on='case_id', how='left')
print(f'Variant rank added. Range: {feats["variant_rank"].min()} – {feats["variant_rank"].max()}')

Variant rank added. Range: 1 – 1478


In [10]:
# ── Save feature table ─────────────────────────────────────────────
feats.to_csv(TABLES_OUT / 'features.csv', index=False)
print(f'Feature table: {feats.shape[0]} rows × {feats.shape[1]} columns')
print()
print(feats.dtypes.to_string())

Feature table: 7065 rows × 25 columns

case_id                        object
duration_days                 float64
duration_bucket              category
is_long                         int64
n_events                        int64
n_rejections                    int64
n_reminders                     int64
n_approvals                     int64
has_rfp                         int64
start_activity                 object
end_activity                   object
starts_with_trip                int64
ends_with_payment               int64
ends_with_reminder              int64
permit_approval_days          float64
approval_to_trip_days         float64
trip_duration_days            float64
decl_to_payment_days          float64
compliance                     object
case:OrganizationalEntity      object
case:RequestedBudget          float64
case:TotalDeclared            float64
case:Overspent                  int64
budget_utilisation            float64
variant_rank                    int64


## 2. Prepare modelling dataset

Encode categoricals, impute missing values, split train/test.

In [11]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import roc_auc_score, accuracy_score, classification_report
from sklearn.pipeline        import Pipeline
from sklearn.impute          import SimpleImputer
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Select modelling features — exclude target leakage columns
EXCLUDE = ['case_id','duration_days','duration_bucket','is_long',
           'start_activity','end_activity','variant']  # high-cardinality strings

model_df = feats.copy()

# Encode categoricals
for col in ['case:OrganizationalEntity','compliance']:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))

feature_cols = [c for c in model_df.columns if c not in EXCLUDE]
X = model_df[feature_cols].copy()
y = model_df['is_long'].values

# Impute then check
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Long-duration rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}')
print(f'Features used: {len(feature_cols)}')
print(feature_cols)

Train: 5652 | Test: 1413
Long-duration rate — train: 0.333  test: 0.333
Features used: 19
['n_events', 'n_rejections', 'n_reminders', 'n_approvals', 'has_rfp', 'starts_with_trip', 'ends_with_payment', 'ends_with_reminder', 'permit_approval_days', 'approval_to_trip_days', 'trip_duration_days', 'decl_to_payment_days', 'compliance', 'case:OrganizationalEntity', 'case:RequestedBudget', 'case:TotalDeclared', 'case:Overspent', 'budget_utilisation', 'variant_rank']


## 3. Fit and evaluate three classifiers

In [12]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, feature_names):
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)

    # 5-fold CV AUC on training set
    cv_auc = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='roc_auc').mean()

    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'  Test accuracy : {acc:.4f}')
    print(f'  Test ROC-AUC  : {auc:.4f}')
    print(f'  CV ROC-AUC    : {cv_auc:.4f}  (5-fold, train set)')
    print(f'\n  Classification report:')
    print(classification_report(y_te, y_pred, target_names=['not long','long'], digits=3))

    # Feature importances
    if hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
    elif hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        importances = np.zeros(len(feature_names))

    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    fi_norm = (fi / fi.sum() * 100).round(2)
    print(f'\n  Top 10 features:')
    print(fi_norm.head(10).to_string())

    return {
        'model': model, 'name': name,
        'acc': acc, 'auc': auc, 'cv_auc': cv_auc,
        'y_proba': y_proba,
        'feature_importance': fi_norm,
    }

print('Evaluating models...')

Evaluating models...


In [13]:
# Logistic Regression — needs scaled features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
res_lr = evaluate('Logistic Regression', lr, X_train_sc, y_train,
                   X_test_sc, y_test, X_train.columns.tolist())


  Logistic Regression
  Test accuracy : 0.9306
  Test ROC-AUC  : 0.9635
  CV ROC-AUC    : 0.9569  (5-fold, train set)

  Classification report:
              precision    recall  f1-score   support

    not long      0.941     0.956     0.948       942
        long      0.910     0.879     0.894       471

    accuracy                          0.931      1413
   macro avg      0.925     0.918     0.921      1413
weighted avg      0.930     0.931     0.930      1413


  Top 10 features:
approval_to_trip_days    26.45
n_reminders              18.82
trip_duration_days       16.33
decl_to_payment_days      7.32
n_events                  6.40
ends_with_reminder        4.91
n_approvals               4.88
permit_approval_days      4.17
starts_with_trip          3.40
ends_with_payment         3.08


In [14]:
# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                             random_state=42, n_jobs=-1)
res_rf = evaluate('Random Forest', rf, X_train, y_train,
                   X_test, y_test, X_train.columns.tolist())


  Random Forest
  Test accuracy : 0.9108
  Test ROC-AUC  : 0.9612
  CV ROC-AUC    : 0.9562  (5-fold, train set)

  Classification report:
              precision    recall  f1-score   support

    not long      0.903     0.970     0.936       942
        long      0.930     0.792     0.856       471

    accuracy                          0.911      1413
   macro avg      0.917     0.881     0.896      1413
weighted avg      0.912     0.911     0.909      1413


  Top 10 features:
approval_to_trip_days    37.40
n_reminders              19.01
ends_with_reminder       10.50
case:TotalDeclared        4.41
ends_with_payment         4.26
trip_duration_days        4.09
n_events                  3.64
decl_to_payment_days      3.44
case:RequestedBudget      3.38
variant_rank              2.77


In [15]:
# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
res_xgb = evaluate('XGBoost', xgb_model, X_train, y_train,
                    X_test, y_test, X_train.columns.tolist())


  XGBoost
  Test accuracy : 0.9321
  Test ROC-AUC  : 0.9744
  CV ROC-AUC    : 0.9677  (5-fold, train set)

  Classification report:
              precision    recall  f1-score   support

    not long      0.933     0.967     0.950       942
        long      0.929     0.862     0.894       471

    accuracy                          0.932      1413
   macro avg      0.931     0.915     0.922      1413
weighted avg      0.932     0.932     0.931      1413


  Top 10 features:
ends_with_reminder       57.669998
n_reminders              15.600000
approval_to_trip_days     7.190000
decl_to_payment_days      2.260000
trip_duration_days        2.130000
ends_with_payment         1.780000
n_events                  1.450000
n_approvals               1.320000
variant_rank              1.150000
n_rejections              1.150000


## 4. Model comparison — ROC curves

In [16]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'Logistic Regression': '#2c7bb6', 'Random Forest': '#1a9641', 'XGBoost': '#d7191c'}

for res in [res_lr, res_rf, res_xgb]:
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=colors[res['name']], linewidth=2,
            label=f"{res['name']} (AUC = {res['auc']:.3f})")

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Long-Duration Case Prediction')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'predictive_roc_curves.png', dpi=150)
plt.show()
print('Saved predictive_roc_curves.png')

Saved predictive_roc_curves.png


## 5. Feature importance comparison

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, res, color in zip(axes,
                           [res_lr, res_rf, res_xgb],
                           ['#2c7bb6','#1a9641','#d7191c']):
    top10 = res['feature_importance'].head(10)
    ax.barh(top10.index[::-1], top10.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Importance (% of total)')
    ax.set_title(res['name'], fontsize=11)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Top 10 Feature Importances by Model', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'predictive_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved predictive_feature_importance.png')

Saved predictive_feature_importance.png


## 6. Model comparison summary table

In [18]:
summary = pd.DataFrame([
    {'model': r['name'], 'test_accuracy': r['acc'],
     'test_roc_auc': r['auc'], 'cv_roc_auc': r['cv_auc']}
    for r in [res_lr, res_rf, res_xgb]
]).round(4)
summary.to_csv(TABLES_OUT / 'predictive_model_summary.csv', index=False)
print(summary.to_string(index=False))

# Save all feature importances
fi_all = pd.concat([
    res['feature_importance'].rename(res['name'])
    for res in [res_lr, res_rf, res_xgb]
], axis=1)
fi_all.to_csv(TABLES_OUT / 'predictive_feature_importance.csv')
print('\nTop 10 features — all models:')
print(fi_all.head(10).round(2).to_string())

              model  test_accuracy  test_roc_auc  cv_roc_auc
Logistic Regression         0.9306        0.9635      0.9569
      Random Forest         0.9108        0.9612      0.9562
            XGBoost         0.9321        0.9744      0.9677

Top 10 features — all models:
                       Logistic Regression  Random Forest    XGBoost
approval_to_trip_days                26.45          37.40   7.190000
n_reminders                          18.82          19.01  15.600000
trip_duration_days                   16.33           4.09   2.130000
decl_to_payment_days                  7.32           3.44   2.260000
n_events                              6.40           3.64   1.450000
ends_with_reminder                    4.91          10.50  57.669998
n_approvals                           4.88           1.67   1.320000
permit_approval_days                  4.17           1.80   1.110000
starts_with_trip                      3.40           0.08   1.080000
ends_with_payment                  

## 7. Calibration check — predicted probability vs actual long-duration rate

In [19]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'Logistic Regression': '#2c7bb6', 'Random Forest': '#1a9641', 'XGBoost': '#d7191c'}

for res in [res_lr, res_rf, res_xgb]:
    prob_true, prob_pred = calibration_curve(y_test, res['y_proba'], n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', linewidth=1.5,
            color=colors[res['name']], label=res['name'])

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual long-duration rate)')
ax.set_title('Calibration Curves — Long-Duration Prediction')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'predictive_calibration.png', dpi=150)
plt.show()
print('Saved predictive_calibration.png')

Saved predictive_calibration.png


## 8. Limitations and next steps

**Retrospective features:** All features use complete case history. This inflates performance
compared to what would be achievable at prediction time. Features like `n_rejections`,
`n_approvals`, `decl_to_payment_days`, and `ends_with_payment` are unavailable until
the case is finished — they describe the outcome, not the drivers.

**Next steps for a genuine predictive model:**
1. Restrict features to a fixed prefix (first k events only)
2. Use remaining time prediction (regression) rather than binary classification
3. Apply SHAP values for more robust feature attribution
4. Handle right-censored cases from the 2019 tail separately
5. Validate with temporal cross-validation (train on 2017, test on 2018)

In [20]:
print('=== PREDICTIVE ANALYTICS SUMMARY ===')
print(f'Target   : is_long (duration > {t67:.1f} days = top tertile)')
print(f'Train set: {len(X_train)} cases | Test set: {len(X_test)} cases')
print(f'Features : {len(feature_cols)}')
print()
print(summary.to_string(index=False))
print()
best = summary.loc[summary['test_roc_auc'].idxmax(), 'model']
print(f'Best model by ROC-AUC: {best}')
print()
print('Top 5 features agreed across RF and XGBoost:')
rf_top5  = set(res_rf['feature_importance'].head(5).index)
xgb_top5 = set(res_xgb['feature_importance'].head(5).index)
print(rf_top5 & xgb_top5)

=== PREDICTIVE ANALYTICS SUMMARY ===
Target   : is_long (duration > 101.1 days = top tertile)
Train set: 5652 cases | Test set: 1413 cases
Features : 19

              model  test_accuracy  test_roc_auc  cv_roc_auc
Logistic Regression         0.9306        0.9635      0.9569
      Random Forest         0.9108        0.9612      0.9562
            XGBoost         0.9321        0.9744      0.9677

Best model by ROC-AUC: XGBoost

Top 5 features agreed across RF and XGBoost:
{'n_reminders', 'ends_with_reminder', 'approval_to_trip_days'}
